# Persona-Conditioned Trolley Debate Pipeline

## 0. Setup

Loads `.env`, validates that all three transcripts are present and meet the 2,000-word minimum, and creates the output directories. If anything is missing this cell raises a clear error before any API calls happen.

In [ ]:
%load_ext autoreload
%autoreload 2

from src import config
from src.client import OpenRouterClient, load_transcript

config.ensure_dirs()

print(f'OpenRouter key set: {bool(config.OPENROUTER_API_KEY)}')
print(f'Models: {config.MODELS}')
print(f'Temperature: {config.TEMPERATURE}')
print()
for p in config.PERSONAS:
    text = load_transcript(p)
    print(f'{p}: {len(text.split()):,} words ({config.PERSONA_LEANING[p]})')

client = OpenRouterClient()
print('\nClient initialised.')

## 1. Solo verdicts (Phase 3)

27 calls: 3 personas × 3 models × 3 scenarios. Each call writes one JSON file to `data/solo/` and one line to `data/log.jsonl`. Re-running this cell skips files that already exist, so partial failures are recoverable.

In [ ]:
from src.generate import run_solo_all

solo_results = run_solo_all(client)
print(f'\n{len(solo_results)} solo records on disk.')

## 2. Within-model debates (Phase 4)

9 debates: 3 models × 3 scenarios, each with 3 rounds (Opening → Response → Final) across the 3 persona instances. Output: `data/debate/debate_<model>_<scenario>.json`, with all rounds embedded.

In [ ]:
from src.debate import run_debate_all

debate_results = run_debate_all(client)
print(f'\n{len(debate_results)} debate arenas on disk.')

## 3. Quantitative metrics (Phase 5, no participant input needed)

Action rate, mean confidence, cross-model agreement, stance stability across debate rounds, refusal counts. Written to `results/metrics.json`.

In [ ]:
import pandas as pd
from src import metrics

df_solo = metrics.solo_dataframe()
df_debate = metrics.debate_dataframe()

print('Solo records:', len(df_solo))
print('Debate records:', len(df_debate))
df_solo.head()

In [ ]:
print('Action rate per (persona, model, scenario):')
metrics.action_rate_table(df_solo)

In [ ]:
print('Mean confidence per (persona, model, scenario):')
metrics.confidence_table(df_solo)

In [ ]:
print('Cross-model agreement per (persona, scenario):')
metrics.cross_model_agreement(df_solo)

In [ ]:
print('Stance stability across debate rounds:')
metrics.stance_stability(df_debate)

In [ ]:
print('Refusal summary:')
metrics.refusal_summary(df_solo, df_debate)

In [ ]:
summary = metrics.compute_all(write_json=True)
print(f'Wrote results/metrics.json — {summary["n_solo_records"]} solo, {summary["n_debate_records"]} debate records.')

## 4. Participant fidelity ratings (Phase 5, in-the-loop)

Each participant runs **the cell for their own persona only**. The form shows them the three trolley scenarios, asks for their own verdict + confidence (the verdict-match ground truth), then displays the three model arguments produced for their persona and asks for a 1–5 agreement rating.

Saves to `data/ratings/ratings_participant_<n>.json`. Re-opening the form rehydrates prior selections so participants can resume.

In [ ]:
from src.ratings import display_form

display_form('participant_1')

In [ ]:
from src.ratings import display_form

display_form('participant_2')

In [ ]:
from src.ratings import display_form

display_form('participant_3')

## 5. Verdict-match and fidelity aggregation

Run after **all three** participants have submitted their ratings. Produces:

- **Verdict-match**: for each (persona × model × scenario), does the model's solo verdict match the participant's self-report?
- **Fidelity table**: the participant's 1–5 agreement scores aggregated by (persona × model × scenario).

In [ ]:
from src import ratings

vm = ratings.verdict_match_table()
if vm is None:
    print('No participant ratings yet — run section 4 first.')
else:
    print('Verdict-match (model verdict vs. participant self-report):')
    display(vm)
    print('\nVerdict-match rate by model:')
    display(vm.groupby('model')['match'].mean().rename('match_rate').to_frame())

In [ ]:
fid = ratings.fidelity_summary()
if fid is None:
    print('No participant ratings yet — run section 4 first.')
else:
    print('1–5 agreement scores:')
    display(fid)
    print('\nMean agreement by (model, scenario):')
    display(fid.groupby(['model', 'scenario'])['agreement_1_to_5'].mean().rename('mean_agreement').to_frame())

## 6. Sanity check: log.jsonl line count

Should be 27 (solo) + 27 (debate × 3 rounds × 3 personas = 81 if all 9 debates are in) = up to 108 lines after a complete run.

In [ ]:
from src import config

if config.LOG_PATH.exists():
    n_lines = sum(1 for _ in config.LOG_PATH.open('r', encoding='utf-8'))
    print(f'{n_lines} lines in {config.LOG_PATH}')
else:
    print('No log file yet.')